# nano Beatrix — the group read: what each path block carries at the end

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pillar1g.ipynb)

**The question**: In the stage with the path in the ninth heads of blocks 1-3 (`nano_pillar1c_lane1.ipynb`) every model learned its lookup in one of the three path heads (block 3 on seeds 1-4, block 2 on seeds 5-6), and at the end the other two path heads hold little of it. With the path in block 3's ninth head alone (`nano_pillar1d_lane1.ipynb`) the path took the lookup on four seeds of six: on two, the model's own lookup head formed first and the path was left unused. So the three path heads matter in training. Do they matter at the end? This notebook switches the path off in one path block at a time, and in two at a time, in the six finished models and reads the held-out text each way; the six models with the path in every block's ninth head (`nano_pillar1b_lane1.ipynb`) are read one block at a time. Nothing trains.

| models | the path switched off in | reads |
|---|---|---|
| the path in blocks 1-3, seeds 1-6 | every subset of blocks 1, 2 and 3 | 48 |
| the path in every block, seeds 1-6 | no block, each block alone, all six | 48 |

A block whose path is switched off has its ninth head on the block's own rotation instead; the trained weights are unchanged. With no block off the read is each model's stored text read, and with every path block off its stored path-off read, bit for bit: the notebook checks both before it ships anything. Each read carries the held-out bits a byte (the trainer's 128 held-out documents, 4,096 bytes each) and their split at the positions whose unit path occurred earlier in the row against the rest.

**The cells**: C1 pins the trainer and holds the decision block; C2 writes the held-out file and checks the twelve models and their stored reads on [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments); C3 reads each model (a model whose record is already on the Hub is skipped) and ships its record to `bed/reports/group/`; C4 renders the table. The reads take about a second each on the G4; the downloads take most of the time.

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on, for the records' upload (without it the reads run and stay in `/content/out`). Run C1, then Runtime > Restart session and run all. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/rotary_scale.py`: `block_read`; `tests/test_block_read.py`).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import itertools, os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.96" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 96), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU (the stored reads were taken in bf16 on the card)'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the shipper reads it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out file, the models, the records
REPO = 'AbstractPhil/geolip-rotary-experiments'      # the models and their stored reads are here; the records ship here
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
PIECE = {'early': 'nhpn16b123', 'every': 'nhpn16'}     # the name's piece: the path in blocks 1-3's ninth heads; in every block's
SERIES = {'early': 'pillar1c', 'every': 'pillar1b'}    # their series
TAG = lambda a, s: f'alhena-both_s{s}{TAIL(24000).replace("_askany_", "_askany_p75d_")}_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_{PIECE[a]}_ahdigit1_{SERIES[a]}'
SETS = {'early': [list(o) for k in range(4) for o in itertools.combinations((1, 2, 3), k)],   # every subset of blocks 1-3 (eight reads)
        'every': None}                                 # the default: no block, each block alone, all six (eight reads)
LABEL = {'early': 'the path in blocks 1-3', 'every': 'the path in every block'}
MODELS = [(f'{LABEL[a]}, seed {s}', a, s, TAG(a, s)) for a in ('early', 'every') for s in range(1, 7)]
TEXT_DOCS, TEXT_MB = 128, 16                           # the stored text reads: the trainer's 128 held-out documents in batches of 16
GROUP_DIR = 'bed/reports/group'                        # this notebook's records (series pillar1g)
TEXT_DIR = 'bed/reports/text'                          # the stored text reads (the parity anchors)
ROOM_DIR = 'bed/reports/room'                          # the stored room looks (each path head's share of attention on the continuation)
print('decision block:', len(MODELS), 'models |', {a: (len(v) if v else 'no block, each alone, all') for a, v in SETS.items()},
      '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 — THE DATA AND THE MODELS: the held-out head (the trainer's 2,048 reserved fineweb-edu rows), and the twelve models with their
# weights, their ninth heads, their stored text reads and room looks on the Hub.
import json, os, time
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from geolip.nano_beatrix import bed_corpus
from geolip.nano_beatrix import room_read as rr
from geolip.nano_beatrix import rotary_scale as rs
HELD = bed_corpus.write_heldout(f'{OUT}/bed')
print('held-out head', os.path.basename(HELD), round(os.path.getsize(HELD) / 1e6, 1), 'MB')
api = HfApi(token=HF_TOKEN)
hub = set(api.list_repo_files(REPO))
need = lambda t: (f'bed/weights/{t}/model.safetensors', f'bed/weights/{t}/extra_head.safetensors', f'bed/weights/{t}/scale.json',
                  f'{TEXT_DIR}/{rs.text_name(t)}.json', f'{ROOM_DIR}/{rr.room_name(t)}.json')
missing = [lab for lab, a, s, t in MODELS if not all(f in hub for f in need(t))]
assert not missing, ('models or their stored reads are not on the Hub', missing)
before = [lab for lab, a, s, t in MODELS if f'{GROUP_DIR}/{rs.group_name(t)}.json' in hub]
print(f'the {len(MODELS)} models with their text reads and room looks on {REPO}: {len(MODELS)} of {len(MODELS)}; records there before this session: {len(before)}')


In [ ]:
# C3 — THE READS: each model fetched from the Hub and read with the path off in each set (rotary_scale.block_read), the reads with no
# block off and with every path block off checked against the model's stored text read bit for bit, the record shipped to
# bed/reports/group/ (a background shipper with retries). A model whose record is on the Hub or in OUT/group/ is not read again.
import shutil
lship = rs.Shipper(REPO) if HF_TOKEN else None
os.makedirs(f'{OUT}/group', exist_ok=True)
todo = [m for m in MODELS if f'{GROUP_DIR}/{rs.group_name(m[3])}.json' not in hub]
t0 = time.time()
for i, (lab, a, s, tag) in enumerate(todo):
    out = f'{OUT}/group/{rs.group_name(tag)}.json'
    if not os.path.exists(out):                       # a record made here before a restart: no download, its check again
        snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    rec = rs.block_read(f'{OUT}/w/bed/weights/{tag}', out, HELD, sets=SETS[a], n_docs=TEXT_DOCS, mb=TEXT_MB, dev='cuda', log=lambda m: None)
    tr = json.load(open(hf_hub_download(REPO, f'{TEXT_DIR}/{rs.text_name(tag)}.json', local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    by = {tuple(r['off']): r for r in rec['reads']}
    full = tuple(rec['path']['blocks'])
    exact = (by[()]['bits'] == tr['bits'] and by[()]['by_repeat'] == tr['by_repeat'] and by[full]['bits'] == tr['path_off']['bits']
             and by[full]['by_repeat'] == tr['path_off']['by_repeat'])
    if not exact:
        os.replace(out, out + '.bad')                  # kept for the look, never shipped
        raise AssertionError((lab, 'the reads at the ends are not the stored text read: the instrument first',
                              by[()]['bits'], tr['bits'], by[full]['bits'], tr['path_off']['bits']))
    if lship:
        lship.submit(f'{OUT}/group', GROUP_DIR)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)   # the session's disk holds one model at a time
    el = time.time() - t0
    print(f"{lab}: no block off {by[()]['bits']:.4f}; " + ', '.join(f"off {'+'.join(map(str, o))} {r['bits'] - by[()]['bits']:+.4f}"
          for o, r in by.items() if o) + f" | {i + 1} of {len(todo)} read, {el / 60:.1f} min spent, about {el / (i + 1) * (len(todo) - i - 1) / 60:.1f} min left", flush=True)
if lship:
    print('the records shipped' if lship.drain() else 'the records still retrying in the background')


In [ ]:
# C4 — THE TABLE (any time; the records made here or read from the Hub): for each model the cost of switching the path off in each
# set (the held-out bits against no block off), the lookup's block (the path head with the largest share of attention on the
# continuation in the stored room look, its share beside it), and the share of the whole toggle that block carries alone.
from IPython.display import Markdown, display

def _rec(local, hub_path):
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))
    try:
        return json.load(open(hf_hub_download(REPO, hub_path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    except Exception:
        return None

for a in ('early', 'every'):
    rows, head = [], None
    for lab, a2, s, tag in MODELS:
        if a2 != a:
            continue
        g = _rec(f'{OUT}/group/{rs.group_name(tag)}.json', f'{GROUP_DIR}/{rs.group_name(tag)}.json')
        rm = _rec(f'{OUT}/hub/{ROOM_DIR}/{rr.room_name(tag)}.json', f'{ROOM_DIR}/{rr.room_name(tag)}.json')
        if g is None or rm is None:
            continue
        by = {tuple(r['off']): r['bits'] for r in g['reads']}
        base, full = by[()], tuple(g['path']['blocks'])
        m = [b[8]['m'] for b in rm['look']['blocks']]
        L = max(full, key=lambda b: m[b - 1])
        sets = [o for o in by if o]
        if head is None:
            head = ['| seed | ' + ' | '.join('off ' + '+'.join(map(str, o)) for o in sets) + " | the lookup's block (its share) | that block alone, of the whole toggle |",
                    '|---' * (len(sets) + 3) + '|']
        rows.append(f'| {s} | ' + ' | '.join(f'{by[o] - base:+.4f}' for o in sets) + f' | {L} ({m[L - 1]:.3f}) | {(by[(L,)] - base) / (by[full] - base):.2f} |')
    if rows:
        display(Markdown(f'**{LABEL[a]}** (held-out bits a byte against no block off)\n\n' + '\n'.join(head + rows)))
